Basic model run, no optimization yet

In [ ]:
import os, torch
import matplotlib.pyplot as plt
import spintorch
import numpy as np
from src.cfg import load_configs
from src.geometry import make_wavegeom
from src.sources import build_sources, temporal_envelope
from src.readouts_legacy import build_disk_probes
from src.solver_only import solve_once

def main():
    cfg = load_configs(".") 
    assert cfg.get("run_mode","solver_only") == "solver_only", \
        "Set run_mode to 'solver_only' in configs/sim_defaults.json"

    dev = torch.device(cfg.get("device","cuda"))
    print("Running solver-only on", dev)

    # I/O
    base = cfg["io"]["basedir"]
    plotdir = os.path.join("runs","plots", base); os.makedirs(plotdir, exist_ok=True)
    savedir = os.path.join("runs","models", base); os.makedirs(savedir, exist_ok=True)

    # geometry
    geom = make_wavegeom(cfg)
    nx, ny = cfg["grid"]["nx"], cfg["grid"]["ny"]
    dx, dy = cfg["grid"]["dx_m"], cfg["grid"]["dy_m"]

    # sources & probes
    src = build_sources(cfg, nx, ny, dx, dy)
    probes = build_disk_probes(cfg, nx, ny)

    # solver
    dt = cfg["time"]["dt_s"]; timesteps = cfg["time"]["timesteps"]
    model = spintorch.MMSolver(geom, dt, src, probes).to(dev)
    model.retain_history = bool(cfg.get("save_history", True))

    # temporal drive
    X, t = temporal_envelope(cfg, dt, timesteps, dev)
    INPUTS = X.repeat(1, 1, len(src))   # [1, T, Nsrc]

    # Optional: baseline assertion there are no extra fields
    try:
        from src.analysis_helpers import assert_baseline_no_weights
        assert_baseline_no_weights(model)
    except Exception:
        pass

    # Plot temporal pulse (for the run log)
    t_ns = t.squeeze().cpu().numpy()*1e9
    env = X.squeeze().detach().cpu().numpy()
    plt.figure(); plt.plot(t_ns, env); plt.xlabel("Time (ns)"); plt.ylabel("B_exc (T)")
    plt.title("Temporal excitation (solver-only)"); plt.grid(True)
    plt.savefig(os.path.join(plotdir, "temporal_envelope_solver_only.png"), dpi=300)

    # Run once with the pulse
    _ = solve_once(model, INPUTS, savedir, plotdir, cfg, tag="pulse")

    # (Optional) Run once with zero drive for a dark reference
    X0 = torch.zeros_like(X)
    INPUTS0 = X0.repeat(1, 1, len(src))
    _ = solve_once(model, INPUTS0, savedir, plotdir, cfg, tag="dark")

if __name__ == "__main__":
    main()


FFT and mode analysis

In [3]:
import os, torch
import matplotlib.pyplot as plt
import spintorch
import numpy as np
from src.cfg import load_configs
from src.geometry import make_wavegeom
from src.sources import build_sources, temporal_envelope
from src.readouts_legacy import build_disk_probes
from src.solver_only import solve_once

# --- utilities ---
def hann(n): return 0.5 - 0.5*np.cos(2*np.pi*np.arange(n)/(n-1))

def P_kt(k, t):
    kt = np.clip(np.abs(k)*t, 1e-12, None)
    return (1.0 - np.exp(-kt))/kt

def omega_DE(k, H, Ms, gamma, t):
    # Damon–Eshbach (k ⟂ M)
    mu0 = 4e-7*np.pi
    P = P_kt(k, t)
    return gamma*mu0*np.sqrt((H + Ms*(1 - P))*(H + Ms*P))

def omega_BV(k, H, Ms, gamma, t):
    # Backward Volume (k ∥ M)
    mu0 = 4e-7*np.pi
    P = P_kt(k, t)
    return gamma*mu0*np.sqrt(H*(H + Ms*P))

def extract_line_mz(mz_full, x_index=None, x_span=None):
    """mz_full: [T, nx, ny] or [T, ny] → returns [T, ny] (CPU, float64)"""
    if mz_full.ndim == 3:
        if x_span is not None:
            x0, x1 = x_span
            data = mz_full[:, x0:x1, :].mean(dim=1)
        else:
            xi = int(mz_full.shape[1]//2) if x_index is None else x_index
            data = mz_full[:, xi:xi+1, :].mean(dim=1)
    elif mz_full.ndim == 2:
        data = mz_full
    else:
        raise ValueError("mz_full must be [T, ny] or [T, nx, ny].")
    return data.detach().cpu().double()

# def main():
cfg = load_configs(".") 
assert cfg.get("run_mode","solver_only") == "solver_only", \
    "Set run_mode to 'solver_only' in configs/sim_defaults.json"

dev = torch.device(cfg.get("device","cuda"))
print("Running solver-only on", dev)

# I/O
base = cfg["io"]["basedir"]
plotdir = os.path.join("runs","plots", base); os.makedirs(plotdir, exist_ok=True)
savedir = os.path.join("runs","models", base); os.makedirs(savedir, exist_ok=True)

# geometry
geom = make_wavegeom(cfg)
nx, ny = cfg["grid"]["nx"], cfg["grid"]["ny"]
dx, dy = cfg["grid"]["dx_m"], cfg["grid"]["dy_m"]

# sources & probes
src = build_sources(cfg, nx, ny, dx, dy)
probes = build_disk_probes(cfg, nx, ny)

# solver
dt = cfg["time"]["dt_s"]; timesteps = cfg["time"]["timesteps"]
model = spintorch.MMSolver(geom, dt, src, probes).to(dev)
model.retain_history = bool(cfg.get("save_history", True))

# temporal drive
X, t = temporal_envelope(cfg, dt, timesteps, dev)
INPUTS = X.repeat(1, 1, len(src))   # [1, T, Nsrc]

# Optional: baseline assertion there are no extra fields
try:
    from src.analysis_helpers import assert_baseline_no_weights
    assert_baseline_no_weights(model)
except Exception:
    pass

# Plot temporal pulse (for the run log)
t_ns = t.squeeze().cpu().numpy()*1e9
env = X.squeeze().detach().cpu().numpy()
plt.figure(); plt.plot(t_ns, env); plt.xlabel("Time (ns)"); plt.ylabel("B_exc (T)")
plt.title("Temporal excitation (solver-only)"); plt.grid(True)
plt.savefig(os.path.join(plotdir, "temporal_envelope_solver_only.png"), dpi=300)

# Run once with the pulse 3.5e-12
_ = solve_once(model, INPUTS, savedir, plotdir, cfg, tag="pulse")

# (Optional) Run once with zero drive for a dark reference
X0 = torch.zeros_like(X)
INPUTS0 = X0.repeat(1, 1, len(src))
_ = solve_once(model, INPUTS0, savedir, plotdir, cfg, tag="dark")

T_total = len(model.m_history)
mz_full = torch.stack(model.m_history, 1)[0, :, 2, ] - model.m0[0, 2, ].unsqueeze(0).cpu()  # [T, nx, ny]
mz_line = extract_line_mz(mz_full, x_span=(mz_full.shape[1]//2 - 2, mz_full.shape[1]//2 + 3))  # [T, ny]

# params
dt = cfg["time"]["dt_s"]
dx = cfg["grid"]["dx_m"]; dy = cfg["grid"]["dy_m"]
nx = cfg["grid"]["nx"]; ny = cfg["grid"]["ny"]

# --- window + zero pad ---
win_t = hann(mz_line.shape[0])[:, None]
win_x = hann(nx)[None, :]; win_y = hann(ny)[None, :]
sig_x = (mz_line.numpy() * (win_t*win_x)); sig_y = (mz_line.numpy() * (win_t*win_y))

zpad_t, zpad_x, zpad_y = 2, 2, 2
NT_x = int(2**np.ceil(np.log2(sig_x.shape[0]*zpad_t)))
NT_y = int(2**np.ceil(np.log2(sig_y.shape[0]*zpad_t)))
NYp_x= int(2**np.ceil(np.log2(nx*zpad_x)))
NYp_y= int(2**np.ceil(np.log2(ny*zpad_y)))

# --- FFT: time → freq, then x → kx ---
SIG_fx = np.fft.rfft(sig_x, n=NT_x, axis=0)                # [Nf, ny]
freqs_x = np.fft.rfftfreq(NT_x, d=dt)                      # Hz
SIG_fk = np.fft.fftshift(np.fft.fft(SIG_fx, n=NYp_x, axis=1), axes=1)
kx = np.fft.fftshift(np.fft.fftfreq(NYp_x, d=dx)) * 2*np.pi  # rad/m
PSD_x = np.abs(SIG_fk)**2

# --- FFT: time → freq, then y → ky ---
SIG_fy = np.fft.rfft(sig_y, n=NT_y, axis=0)                # [Nf, ny]
freqs_y = np.fft.rfftfreq(NT_y, d=dt)                      # Hz
SIG_fk = np.fft.fftshift(np.fft.fft(SIG_fy, n=NYp_y, axis=1), axes=1)
ky = np.fft.fftshift(np.fft.fftfreq(NYp_y, d=dy)) * 2*np.pi  # rad/m
PSD_y = np.abs(SIG_fk)**2

# axis in human units
freq_GHz_x = freqs_x/1e9; freq_GHz_y = freqs_y/1e9
kx_per_um = kx/1e6; ky_per_um = ky/1e6

# --- theory overlay ---
Ms   = float(cfg["material"]["Ms_Aperm"])      # A/m
tfilm= float(cfg["grid"]["dz_m"])      # m (adjust to your physical thickness if needed)
B0   = float(cfg["bias"]["B0_T"])      # T
mu0  = 4e-7*np.pi
H0   = B0/mu0                    # A/m
gamma= 1.760859e11               # rad/(s·T)

Running solver-only on cuda
Baseline check passed: uniform film, no Ms/ΔH weights active.


c:\Users\Tojo\spintorch\source.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('x', torch.tensor(x, dtype=torch.int64))
c:\Users\Tojo\spintorch\source.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('y', torch.tensor(y, dtype=torch.int64))
c:\Users\Tojo\lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differe

In [4]:
plt.figure(figsize=(7.6,5.8))
plt.plot(1 - P_kt(kx, tfilm*10), color = "black")
plt.savefig(os.path.join(plotdir,"test.png"), dpi=300)

In [5]:
# --- plot BV k–f with overlay ---

f_theory = omega_BV(kx, H0, Ms, gamma, tfilm)/(2*np.pi)
f_theory_GHz = f_theory/1e9

plt.figure(figsize=(7.6,5.8))
extent=[kx_per_um.min(), kx_per_um.max(), freq_GHz_x.min(), freq_GHz_x.max()]
plt.imshow(10*np.log10(PSD_x.T+1e-20).transpose(), origin='lower', aspect='auto', extent=extent)
plt.plot(kx_per_um, f_theory_GHz, color = "r", lw=2)
plt.axhline(1e-9*omega_BV(0, H0, Ms, gamma, tfilm)/(2*np.pi), color = "black", lw=2)
plt.axhline(1e-9*omega_BV(1e20, H0, Ms, gamma, tfilm)/(2*np.pi), color = "cyan", lw=2)
plt.xlabel(r"$k_y$ (rad / $\mu$m)"); plt.ylabel("Frequency (GHz)")
# plt.ylim(0, 10)
plt.title(f"k-f map + BV theory")
plt.colorbar(label="Power (dB)")
plt.tight_layout(); plt.show()
plt.savefig(os.path.join(plotdir,"BV k-f with overlay.png"), dpi=300)

# --- ridge vs theory error ---
power_per_k = PSD_x.sum(axis=0)
valid = power_per_k >= np.percentile(power_per_k, 75)
peak_idx = PSD_x[:, valid].argmax(axis=0)
f_peak = freq_GHz_x[peak_idx]
k_sel = kx[valid]

f_sel_th = omega_BV(k_sel, H0, Ms, gamma, tfilm)/(2*np.pi)/1e9

rel_err = np.abs(f_peak - f_sel_th)/np.maximum(f_sel_th, 1e-12)
print(f"mean rel. error = {100*rel_err.mean():.2f}%   (pass if < 10%)")
print(f"max  rel. error = {100*rel_err.max():.2f}%   on {valid.sum()} k-columns")

C:\Users\Tojo\AppData\Local\Temp\ipykernel_22120\1564548462.py:16: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.tight_layout(); plt.show()


mean rel. error = 7.91%   (pass if < 10%)
max  rel. error = 17.89%   on 64 k-columns


In [ ]:
f_theory = omega_DE(ky, H0, Ms, gamma, tfilm)/(2*np.pi)
f_theory_GHz = f_theory/1e9

# --- plot DE k–f with overlay ---
plt.figure(figsize=(7.6,5.8))
extent=[ky_per_um.min(), ky_per_um.max(), freq_GHz_y.min(), freq_GHz_y.max()]
plt.imshow(10*np.log10(PSD_y.T+1e-20).transpose(), origin='lower', aspect='auto', extent=extent)
plt.plot(ky_per_um, f_theory_GHz, color = "r", lw=2)
plt.ylim(0, 10)
plt.xlabel(r"$k_y$ (rad / $\mu$m)"); plt.ylabel("Frequency (GHz)")
plt.title(f"k-f map + DE theory")
plt.colorbar(label="Power (dB)")
plt.tight_layout(); plt.show()
plt.savefig(os.path.join(plotdir,"DE k-f with overlay.png"), dpi=300)

# --- ridge vs theory error ---
power_per_k = PSD_y.sum(axis=0)
valid = power_per_k >= np.percentile(power_per_k, 75)
peak_idx = PSD_y[:, valid].argmax(axis=0)
f_peak = freq_GHz_y[peak_idx]
k_sel = ky[valid]
f_sel_th = omega_DE(k_sel, H0, Ms, gamma, tfilm)/(2*np.pi)/1e9

rel_err = np.abs(f_peak - f_sel_th)/np.maximum(f_sel_th, 1e-12)
print(f"mean rel. error = {100*rel_err.mean():.2f}%   (pass if < 10%)")
print(f"max  rel. error = {100*rel_err.max():.2f}%   on {valid.sum()} k-columns")

C:\Users\Tojo\AppData\Local\Temp\ipykernel_22120\376578794.py:13: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.tight_layout(); plt.show()


mean rel. error = 2.71%   (pass if < 10%)
max  rel. error = 6.05%   on 64 k-columns


Fluence (Bt) sweep - nonlinearity check

In [7]:
# ==== Fluence (Bt) sweep: linear scaling + spectral broadening ====
import numpy as np
import torch
import matplotlib.pyplot as plt

def fwhm_3db(freqs, psd):
    """Return 3-dB bandwidth (Hz) around the peak. freqs: 1D, psd: 1D (nonneg)."""
    psd = np.asarray(psd)
    if psd.max() <= 0: return 0.0
    peak = psd.max()
    thr = peak / (10**(3/10))  # -3 dB in linear scale
    # find contiguous region around peak above threshold
    i0 = psd.argmax()
    left = i0
    while left > 0 and psd[left] >= thr: left -= 1
    right = i0
    n = psd.size
    while right < n-1 and psd[right] >= thr: right += 1
    # linear interpolate for edges
    fL = np.interp(thr, [psd[left], psd[left+1]], [freqs[left], freqs[left+1]]) if left < i0 else freqs[left]
    fR = np.interp(thr, [psd[right-1], psd[right]], [freqs[right-1], freqs[right]]) if right > i0 else freqs[right]
    bw = max(fR - fL, 0.0)
    return bw

# Which probe to analyze spectrally (e.g., the "target" probe)
probe_idx = cfg["probes"]["Ndisk"] // 4

Bt_values = np.logspace(-4, 0, 10)  # T, e.g. 1e-4 ... 1e-2
I_total   = []                      # integrated total intensity across probes
I_probe   = []                      # integrated intensity at chosen probe
BW_3dB    = []                      # spectral 3-dB bandwidth (Hz) at chosen probe
spec_list = []                      # store normalized spectra for plotting (optional)

for Bt in Bt_values:
    # build temporal drive at this Bt (scale relative to config Bt)
    X_base, tvec = temporal_envelope(cfg, dt, timesteps, dev)  # X_base is for cfg["source"]["Bt_T"]
    X = (Bt / cfg["source"]["Bt_T"]) * X_base
    INPUTS = X.repeat(1, 1, len(src))  # [1, T, Nsrc]

    with torch.no_grad():
        model.retain_history = False   # we only need probe time-series here
        Y = model(INPUTS)              # expect shape [1, T, Nprobes]
        if Y.ndim == 2:                # if model already summed time, bail (unlikely)
            raise RuntimeError("model(INPUTS) returned [1, Nprobes]; need time-series [1, T, Nprobes].")
        y_ts = Y[0].detach().cpu().numpy()  # [T, Nprobes]

    # time integration (sum over time samples) → intensity per probe
    I_per_probe = y_ts.sum(axis=0)                 # [Nprobes]
    I_total.append(I_per_probe.sum())              # scalar
    I_probe.append(I_per_probe[probe_idx])         # scalar

    # spectrum at chosen probe
    sig = y_ts[:, probe_idx]
    sig = sig - sig.mean()
    spec = np.abs(np.fft.rfft(sig))**2
    freqs = np.fft.rfftfreq(sig.size, d=dt)

    # normalize spectrum for plotting (peak=0 dB)
    spec_db = 10*np.log10(np.maximum(spec/spec.max(), 1e-20))
    spec_list.append((freqs.copy(), spec_db.copy()))

    # 3-dB bandwidth (in Hz)
    BW_3dB.append(fwhm_3db(freqs, spec))

I_total = np.array(I_total)
I_probe = np.array(I_probe)
BW_3dB  = np.array(BW_3dB)

# ---- Scaling plot (total intensity vs Bt) ----
plt.figure(figsize=(5.8,4.4))
plt.loglog(Bt_values, I_total, 'o-', label='Total output')
plt.loglog(Bt_values, I_probe, 's--', label=f'Probe {probe_idx}')
# reference ~ Bt^2 line anchored at first point
ref_k = I_total[0] / (Bt_values[0]**2 + 1e-30)
plt.loglog(Bt_values, ref_k*Bt_values**2, ':', label='~ Bt$^2$ reference')
plt.xlabel("Excitation amplitude Bt (T)")
plt.ylabel("Integrated intensity (arb.)")
plt.title("Scaling vs fluence")
plt.legend(); plt.tight_layout(); plt.show()
plt.savefig(os.path.join(plotdir,"Scaling vs fluence.png"), dpi=300)

# slope estimate in the small-Bt (first 2–3 points) region
subset = slice(0, min(3, len(Bt_values)))
p = np.polyfit(np.log10(Bt_values[subset]), np.log10(I_total[subset]+1e-30), 1)
print(f"Low-fluence slope (Total): {p[0]:.2f}  (expected ≈ 2.0 in linear regime)")
p_probe = np.polyfit(np.log10(Bt_values[subset]), np.log10(I_probe[subset]+1e-30), 1)
print(f"Low-fluence slope (Probe {probe_idx}): {p_probe[0]:.2f}")

# ---- Spectral broadening vs Bt (3-dB bandwidth) ----
plt.figure(figsize=(5.8,4.4))
plt.semilogx(Bt_values, BW_3dB/1e9, 'o-')
plt.xlabel("Bt (T)")
plt.ylabel("3-dB bandwidth (GHz)")
plt.title("Spectral broadening with fluence")
plt.tight_layout(); plt.show()

# (optional) overlay normalized spectra
plt.figure(figsize=(6.8,4.4))
for (f, sdb), Bt in zip(spec_list, Bt_values):
    plt.plot(f/1e9, sdb, label=f"Bt={Bt:.0e} T")
plt.xlabel("Frequency (GHz)"); plt.ylabel("Normalized PSD (dB)")
plt.title(f"Spectra at probe {probe_idx}")
plt.legend(ncol=2, fontsize=8); plt.tight_layout(); plt.show()

C:\Users\Tojo\AppData\Local\Temp\ipykernel_22120\74831841.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.legend(); plt.tight_layout(); plt.show()


Low-fluence slope (Total): 1.99  (expected ≈ 2.0 in linear regime)
Low-fluence slope (Probe 4): 1.99


C:\Users\Tojo\AppData\Local\Temp\ipykernel_22120\74831841.py:95: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.tight_layout(); plt.show()
C:\Users\Tojo\AppData\Local\Temp\ipykernel_22120\74831841.py:103: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.legend(ncol=2, fontsize=8); plt.tight_layout(); plt.show()


phase-space radius - optional

In [ ]:
# # =========================
# # Single-source, uniform film: k-space ring & H0 sweep
# # =========================
# import os, numpy as np, torch, matplotlib.pyplot as plt
# import spintorch
# from src.cfg import load_configs
# from src.geometry import make_wavegeom
# from src.sources import build_sources, temporal_envelope
# from src.readouts import build_disk_probes
# from src.solver_only import solve_once

# def hann(n): 
#     return 0.5 - 0.5*np.cos(2*np.pi*np.arange(n)/(n-1))

# def kmap_2d_at_peak(mhist, dt, dx, dy, comp=2):
#     """
#     Robustly builds a 2D k-space map at the dominant frequency.
#     Accepts:
#       - mhist: list of [batch, 3, nx, ny] tensors (SpinTorch typical), or
#       - a stacked tensor with shape [batch, T, 3, nx, ny] or [T, 3, nx, ny].
#     Returns: f_peak_GHz, KX(rad/m), KY(rad/m), PSD2D [NX, NY]
#     """
#     # ---- normalize to [batch, T, 3, nx, ny] on CPU ----
#     if isinstance(mhist, list):
#         # list length = T, each elem [batch, 3, nx, ny]
#         assert mhist[0].ndim in (4, 3), "Unexpected m_history element shape"
#         if mhist[0].ndim == 4:  # [batch, 3, nx, ny]
#             m = torch.stack(mhist, dim=1).cpu()        # [batch, T, 3, nx, ny]
#         else:  # [3, nx, ny]
#             m = torch.stack(mhist, dim=0).unsqueeze(0).cpu()  # [1, T, 3, nx, ny]
#     else:
#         m = mhist
#         if m.ndim == 5:          # [batch, T, 3, nx, ny]
#             m = m.cpu()
#         elif m.ndim == 4:        # [T, 3, nx, ny] -> add batch
#             m = m.unsqueeze(0).cpu()
#         else:
#             raise ValueError(f"Unexpected stacked history shape: {tuple(m.shape)}")

#     # ---- pick batch 0, subtract equilibrium per-pixel ----
#     # m0 should be [batch, 3, nx, ny]; fall back to first time frame if not available.
#     try:
#         m0 = model.m0.cpu()[0]                        # [3, nx, ny]
#     except Exception:
#         m0 = m[0, 0]                                  # [3, nx, ny] (t=0)

#     # signal: [T, nx, ny] for selected component
#     sig = (m[0, :, comp] - m0[comp].unsqueeze(0)).double().numpy()  # [T, nx, ny]
#     T, nx, ny = sig.shape

#     # ---- temporal FFT: pick dominant frequency bin ----
#     zt = 2
#     NT = 1 << int(np.ceil(np.log2(T * zt)))
#     M_f = np.fft.rfft(sig, n=NT, axis=0)                  # [Nf, nx, ny]
#     freqs = np.fft.rfftfreq(NT, d=dt)
#     ifreq = np.abs(M_f).sum(axis=(1, 2)).argmax()
#     f_peak_GHz = freqs[ifreq] / 1e9

#     # ---- 2D spatial FFT at f_peak ----
#     zx = zy = 2
#     NX = 1 << int(np.ceil(np.log2(nx * zx)))
#     NY = 1 << int(np.ceil(np.log2(ny * zy)))
#     Fxy = np.fft.fftshift(np.fft.fft2(M_f[ifreq], s=(NX, NY)))
#     KX = np.fft.fftshift(np.fft.fftfreq(NX, d=dx)) * 2 * np.pi
#     KY = np.fft.fftshift(np.fft.fftfreq(NY, d=dy)) * 2 * np.pi
#     PSD2D = np.abs(Fxy) ** 2
#     return f_peak_GHz, KX, KY, PSD2D


# def ring_radius(psd2d, KX, KY, nbins=240):
#     kxg, kyg = np.meshgrid(KX, KY, indexing='ij')
#     Kmag = np.sqrt(kxg**2 + kyg**2)  # [NX,NY]
#     P = psd2d
#     kmin, kmax = Kmag.min(), Kmag.max()
#     bins = np.linspace(kmin, kmax, nbins+1)
#     idx = np.digitize(Kmag.ravel(), bins) - 1
#     prof = np.zeros(nbins)
#     for b in range(nbins):
#         sel = (idx==b)
#         if np.any(sel): prof[b] = P.ravel()[sel].mean()
#     kcent = 0.5*(bins[:-1]+bins[1:])
#     k_peak = kcent[np.nanargmax(prof)]
#     return k_peak, (kcent, prof)

# def angular_profile(psd2d, KX, KY, k0, dk_frac=0.1, nbins=180):
#     kxg, kyg = np.meshgrid(KX, KY, indexing='ij')
#     Kmag = np.sqrt(kxg**2 + kyg**2)
#     Theta = np.arctan2(kyg, kxg)   # [-pi, pi]
#     ring = (Kmag >= k0*(1-dk_frac)) & (Kmag <= k0*(1+dk_frac))
#     ang = Theta[ring].ravel()
#     powv= psd2d[ring].ravel()
#     bins = np.linspace(-np.pi, np.pi, nbins+1)
#     which = np.digitize(ang, bins) - 1
#     prof = np.zeros(nbins)
#     for b in range(nbins):
#         sel = (which==b)
#         if np.any(sel): prof[b] = powv[sel].mean()
#     centers = 0.5*(bins[:-1]+bins[1:])
#     return centers, prof

# # ----------------- RUN: single-source, sweep B0 -----------------
# cfg = load_configs(".")
# dev = torch.device(cfg.get("device","cuda"))

# # I/O
# base = cfg["io"]["basedir"]
# plotdir = os.path.join("runs","plots", base); os.makedirs(plotdir, exist_ok=True)
# savedir = os.path.join("runs","models", base); os.makedirs(savedir, exist_ok=True)

# # grid/time
# nx, ny = cfg["grid"]["nx"], cfg["grid"]["ny"]
# dx, dy = cfg["grid"]["dx_m"], cfg["grid"]["dy_m"]
# dt, Tsteps = cfg["time"]["dt_s"], cfg["time"]["timesteps"]

# # field sweep (Tesla)
# B0_center = cfg["bias"]["B0_T"]
# B0_list = np.linspace(0.5*B0_center, 1.5*B0_center, 5)  # e.g., 0.5× ... 1.5×

# kpeaks, fpeaks = [], []

# for i, B0 in enumerate(B0_list):
#     # --- geometry & model (uniform film) ---
#     geom = make_wavegeom({**cfg, "bias": {"B0_T": float(B0)}})  # reuse config, override B0
#     src  = build_sources(cfg, nx, ny, dx, dy)                   # single source recommended
#     probes = build_disk_probes(cfg, nx, ny)
#     model = spintorch.MMSolver(geom, dt, src, probes).to(dev)
#     model.retain_history = True

#     # --- temporal drive (use configured shape & Bt_T) ---
#     X, t = temporal_envelope(cfg, dt, Tsteps, dev)              # [1,T,1]
#     INPUTS = X.repeat(1, 1, len(src))                           # [1,T,Nsrc]

#     # --- run once (solver-only) ---
#     with torch.no_grad():
#         _ = model(INPUTS).sum(dim=1)

#     # --- k-space map at dominant frequency ---
#     fpk, KX, KY, PSD2D = kmap_2d_at_peak(model.m_history, dt, dx, dy, comp=2)
#     k_peak, (kr, rprof) = ring_radius(PSD2D, KX, KY, nbins=240)
#     kpeaks.append(k_peak); fpeaks.append(fpk)

#     # --- plots for this field ---
#     plt.figure(figsize=(6.2, 5.4))
#     extent = [KX.min()/1e6, KX.max()/1e6, KY.min()/1e6, KY.max()/1e6]

#     # Prepare array in dB with a tiny floor
#     arr_db = 10*np.log10(np.maximum(PSD2D.T, 1e-20))

#     # If the image is constant (no contrast), skip colorbar & contour gracefully
#     dr = float(np.nanmax(arr_db) - np.nanmin(arr_db)) if np.isfinite(arr_db).all() else 0.0
#     img = plt.imshow(arr_db, origin='lower', extent=extent, aspect='equal')

#     # Only draw colorbar if we have dynamic range
#     if dr > 1e-6:
#         plt.colorbar(img, label="Power (dB)")

#     # Only draw the ring contour if its level is within the data range
#     kgrid = (np.sqrt(np.add.outer(KX**2, KY**2))/1e6).T
#     if np.isfinite(k_peak) and (k_peak/1e6 >= kgrid.min()) and (k_peak/1e6 <= kgrid.max()):
#         try:
#             plt.contour(kgrid, levels=[k_peak/1e6], colors='w', linewidths=1)
#         except Exception:
#             pass

#     plt.xlabel(r"$k_x$ (rad/μm)"); plt.ylabel(r"$k_y$ (rad/μm)")
#     plt.title(f"k-space @ f≈{fpk:.2f} GHz, B0={B0*1e3:.0f} mT")
#     plt.tight_layout()
#     plt.show()


#     # angular anisotropy on the ring
#     th, aprof = angular_profile(PSD2D, KX, KY, k_peak, dk_frac=0.12, nbins=180)
#     plt.figure(figsize=(6.0,3.6))
#     plt.plot(th*180/np.pi, aprof)
#     plt.xlabel("Angle (deg, 0°=+kx)"); plt.ylabel("Ring power (arb.)")
#     plt.title(f"Angular anisotropy, B0={B0*1e3:.0f} mT"); plt.tight_layout()
#     plt.savefig(os.path.join(plotdir, f"anisotropy_B0_{int(B0*1e3)}mT.png"), dpi=200)
#     plt.show()

# # --- Radius vs field summary ---
# kpeaks = np.array(kpeaks); fpeaks = np.array(fpeaks)
# plt.figure(figsize=(5.8,4.2))
# plt.plot(B0_list*1e3, kpeaks/1e6, "o-")
# plt.xlabel(r"$B_0$ (mT)"); plt.ylabel(r"Ring radius $|k|$ (rad/μm)")
# plt.title("k-space ring radius vs bias field"); plt.tight_layout()
# plt.savefig(os.path.join(plotdir, "k_radius_vs_B0.png"), dpi=200)
# plt.show()

ModuleNotFoundError: No module named 'src.readouts'

In [18]:
import os, torch
import matplotlib.pyplot as plt
import spintorch
import numpy as np
from src.cfg import load_configs
from src.geometry import make_wavegeom
from src.sources import build_sources, temporal_envelope
from src.solver_only import solve_once
from src.readout import ROIIntegrator

# def main():
cfg = load_configs(".") 
assert cfg.get("run_mode","solver_only") == "solver_only", \
    "Set run_mode to 'solver_only' in configs/sim_defaults.json"

dev = torch.device(cfg.get("device","cuda"))
print("Running solver-only on", dev)

# I/O
base = cfg["io"]["basedir"]
plotdir = os.path.join("runs","plots", base); os.makedirs(plotdir, exist_ok=True)
savedir = os.path.join("runs","models", base); os.makedirs(savedir, exist_ok=True)

# geometry
geom = make_wavegeom(cfg)
nx, ny = cfg["grid"]["nx"], cfg["grid"]["ny"]
dx, dy = cfg["grid"]["dx_m"], cfg["grid"]["dy_m"]

# sources & probes
src = build_sources(cfg, nx, ny, dx, dy)
probes = build_disk_probes(cfg, nx, ny)

# solver
dt = cfg["time"]["dt_s"]; timesteps = cfg["time"]["timesteps"]
model = spintorch.MMSolver(geom, dt, src, probes).to(dev)
model.retain_history = bool(cfg.get("save_history", True))

# temporal drive
X, t = temporal_envelope(cfg, dt, timesteps, dev)
INPUTS = X.repeat(1, 1, len(src))   # [1, T, Nsrc]

# Optional: baseline assertion there are no extra fields
try:
    from src.analysis_helpers import assert_baseline_no_weights
    assert_baseline_no_weights(model)
except Exception:
    pass

# Plot temporal pulse (for the run log)
t_ns = t.squeeze().cpu().numpy()*1e9
env = X.squeeze().detach().cpu().numpy()
plt.figure(); plt.plot(t_ns, env); plt.xlabel("Time (ns)"); plt.ylabel("B_exc (T)")
plt.title("Temporal excitation (solver-only)"); plt.grid(True)
plt.savefig(os.path.join(plotdir, "temporal_envelope_solver_only.png"), dpi=300)

# Run once with the pulse
_ = solve_once(model, INPUTS, savedir, plotdir, cfg, tag="pulse")

# (Optional) Run once with zero drive for a dark reference
X0 = torch.zeros_like(X)
INPUTS0 = X0.repeat(1, 1, len(src))
_ = solve_once(model, INPUTS0, savedir, plotdir, cfg, tag="dark")

# mz_full = torch.stack(model.m_history, 1)[0, :, 2, ] - model.m0[0, 2, ].unsqueeze(0).cpu()
print(np.shape(model.m_history))

# self,
#     nx:int, ny:int, dx:float, dy:float,
#     t_start_idx:int, t_end_idx:int,
#     dt_s: float,
#     box_xy: Optional[Tuple[float,float,float,float]] = None,
#     box_idx: Optional[Tuple[int,int,int,int]] = None,
#     normalize_area: bool = True,
#     mx_idx:int = 0, my_idx:int = 1,
#     device: Union[str, torch.device] = "cpu",
#     dtype: torch.dtype = torch.float32,

Running solver-only on cuda
Baseline check passed: uniform film, no Ms/ΔH weights active.


c:\Users\Tojo\spintorch\source.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('x', torch.tensor(x, dtype=torch.int64))
c:\Users\Tojo\spintorch\source.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('y', torch.tensor(y, dtype=torch.int64))
c:\Users\Tojo\lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differe

(600, 1, 3, 100, 100)


In [ ]:
import torch.nn as nn
from src.readout import ROIIntegrator   # your module

nx, ny = cfg["grid"]["nx"], cfg["grid"]["ny"]
dx, dy = cfg["grid"]["dx_m"], cfg["grid"]["dy_m"]
dt     = cfg["time"]["dt_s"]
T      = cfg["time"]["timesteps"]

# define multiple ROIs as index boxes
roi_boxes = [
    (90, 100, 40, 50),   # left ROI
    (90, 100, 60, 70),   # middle ROI
    (90, 100, 80, 90),   # right ROI
]

# build a ModuleList of ROIIntegrator instances
rois = nn.ModuleList([
    ROIIntegrator(nx, ny, dx, dy,
                  t_start_idx=0, t_end_idx=T,
                  dt_s=dt,
                  box_idx=box)
    for box in roi_boxes
])

m_hist = torch.stack(model.m_history, dim=1)  # [B,T,3,nx,ny]

# Apply each ROI to the same movie
outs = [roi(m_hist) for roi in rois]       # list of [B]
outs = torch.stack(outs, dim=1)            # shape [B, Nrois]
print(outs)

tensor([[1.2000e-08, 1.2000e-08, 1.2000e-08]])


In [ ]:
def bandpassed_energy(m_hist, dt, f0_Hz, bw_Hz, box_idx, dx, dy):
    """
    Measure bandpassed energy around f0 via short-time FFT inside a rectangular ROI.

    Parameters
    ----------
    m_hist : torch.Tensor [B, T, 3, nx, ny]
        Full magnetization history.
    dt : float
        Time step [s].
    f0_Hz : float
        Center frequency for bandpass [Hz].
    bw_Hz : float
        Bandwidth [Hz].
    box_idx : (xi_min, xi_max, yi_min, yi_max)
        ROI indices.
    dx, dy : float
        Cell sizes (not directly used here but handy if extending to meters).

    Returns
    -------
    energy : torch.Tensor [B]
        Bandpassed energy in the ROI (per batch).
    """

    # --- extract ROI and components ---
    xi0, xi1, yi0, yi1 = box_idx
    roi = m_hist[..., xi0:xi1, yi0:yi1]       # [B, T, 3, nx_roi, ny_roi]
    roi = roi[:, :, :2].contiguous()          # take mx,my only [B,T,2,nx_roi,ny_roi]
    roi = roi.pow(2).sum(dim=2)               # [B, T, nx_roi, ny_roi]

    # --- average spatially ---
    roi_mean = roi.mean(dim=(-2, -1))         # [B, T]

    # --- STFT parameters ---
    B, T = roi_mean.shape
    win = get_window("hann", T, fftbins=True)
    win = torch.tensor(win, dtype=roi_mean.dtype, device=roi_mean.device)
    sig = roi_mean * win                      # apply Hann window

    # --- FFT along time axis ---
    fft_out = torch.fft.rfft(sig, dim=1)      # [B, Nf]
    freqs = np.fft.rfftfreq(T, d=dt)          # Hz

    # --- find band mask ---
    mask = (freqs >= f0_Hz - bw_Hz/2) & (freqs <= f0_Hz + bw_Hz/2)
    mask = torch.tensor(mask, dtype=torch.bool, device=roi_mean.device)

    # --- energy in band ---
    energy = (fft_out.abs()**2)[:, mask].sum(dim=1)
    return energy

In [ ]:
from src.readout import ROIIntegrator

# after running solver and stacking history
m_hist = torch.stack(model.m_history, dim=1)  # [B,T,3,nx,ny]

# pick one ROI
box = (90, 100, 45, 55)   # indices (xi_min, xi_max, yi_min, yi_max)

energy = bandpassed_energy(
    m_hist, dt=cfg["time"]["dt_s"],
    f0_Hz=4e9, bw_Hz=0.5e9,
    box_idx=box,
    dx=cfg["grid"]["dx_m"], dy=cfg["grid"]["dy_m"]
)
print("Bandpassed energy at f0=4 GHz:", energy.cpu().numpy())


Bandpassed energy at f0=4 GHz: [3.0168613e-12]


adding scatterers

In [16]:
import matplotlib.pyplot as plt

cfg = load_configs(".") 
assert cfg.get("run_mode","solver_only") == "solver_only", \
    "Set run_mode to 'solver_only' in configs/sim_defaults.json"

dev = torch.device(cfg.get("device","cuda"))
print("Running solver-only on", dev)

# I/O
base = cfg["io"]["basedir"]
plotdir = os.path.join("runs","plots", base); os.makedirs(plotdir, exist_ok=True)
savedir = os.path.join("runs","models", base); os.makedirs(savedir, exist_ok=True)

# geometry
geom = make_wavegeom(cfg)

plt.clf()

with torch.no_grad():
    rho = geom.rho.detach().cpu().numpy()
plt.imshow(rho.T, origin='lower', aspect='equal')
plt.title("Ms scale (rho)"); plt.colorbar(); plt.show()
plt.savefig(os.path.join(plotdir, "test.png"), dpi=200)
plt.show()

Running solver-only on cuda


C:\Users\Tojo\AppData\Local\Temp\ipykernel_22120\1367846824.py:23: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.title("Ms scale (rho)"); plt.colorbar(); plt.show()
C:\Users\Tojo\AppData\Local\Temp\ipykernel_22120\1367846824.py:25: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [15]:
cfg.get("scatterers")

{'enabled': True,
 'mode': 'single',
 'Ms_scale': 0.7,
 'diameter_m': 1.5e-07,
 'single_center_m': [5e-06, 5e-06],
 'pitch_m': 4e-07,
 'nx_span_m': 1e-05,
 'ny_span_m': 1e-05}

single scatterer test

In [12]:
import numpy as np
import torch, matplotlib.pyplot as plt

Ms_scales = [1.0, 0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6]
f0 = 5e9  # Hz

nx, ny = cfg["grid"]["nx"], cfg["grid"]["ny"]
dx, dy = cfg["grid"]["dx_m"], cfg["grid"]["dy_m"]
dt, timesteps = cfg["time"]["dt_s"], cfg["time"]["timesteps"]

# Source: Gaussian at left edge
src = build_sources(cfg, nx, ny, dx, dy)
probes = build_disk_probes(cfg, nx, ny)

results = {}
for scale in Ms_scales:
    # --- override scatterer config ---
    cfg_scatt = {
        "enabled": True,
        "mode": "single",
        "Ms_scale": scale,
        "diameter_m": 200e-9,
        "single_center_m": [nx*dx/2, ny*dy/2]  # center
    }
    cfg["scatterers"] = cfg_scatt

    # Geometry
    geom = make_wavegeom(cfg)
    model = spintorch.MMSolver(geom, dt, src, probes).to(dev)
    model.retain_history = True

    # Temporal drive
    X, t = temporal_envelope(cfg, dt, timesteps, dev)
    INPUTS = X.repeat(1, 1, len(src))

    with torch.no_grad():
        u = model(INPUTS).sum(dim=1)  # [batch, Nprobes]

    sig = u[0].cpu().numpy()  # pick probe at transmission side
    freqs = np.fft.rfftfreq(len(sig), d=dt)
    SPEC = np.fft.rfft(sig - sig.mean())

    # Extract amplitude & phase at f0
    idx = np.argmin(np.abs(freqs - f0))
    amp, phase = np.abs(SPEC[idx]), np.angle(SPEC[idx])
    results[scale] = (amp, phase)

# Example run

amps, phases = zip(*[results[s] for s in Ms_scales])

plt.figure()
plt.plot(Ms_scales, amps, 'o-')
plt.xlabel("Ms scale"); plt.ylabel("Transmission amplitude")
plt.title("Scatterer attenuation vs Ms scale")
plt.savefig(os.path.join(plotdir, "Scatterer attenuation vs Ms scale.png"), dpi=200)

plt.figure()
plt.plot(Ms_scales, np.unwrap(phases), 'o-')
plt.xlabel("Ms scale"); plt.ylabel("Phase (rad)")
plt.title("Scatterer phase shift vs Ms scale")
plt.savefig(os.path.join(plotdir, "Scatterer phase shift vs Ms scale.png"), dpi=200)
plt.show()


c:\Users\Tojo\spintorch\source.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('x', torch.tensor(x, dtype=torch.int64))
c:\Users\Tojo\spintorch\source.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('y', torch.tensor(y, dtype=torch.int64))
c:\Users\Tojo\lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differe

In [ ]:
def single_scatterer_test(cfg, Ms_scales, f0_Hz, dev="cuda"):

    # def main():
    cfg = load_configs(".") 
    assert cfg.get("run_mode","solver_only") == "solver_only", \
        "Set run_mode to 'solver_only' in configs/sim_defaults.json"

    dev = torch.device(cfg.get("device","cuda"))
    print("Running solver-only on", dev)

    # I/O
    base = cfg["io"]["basedir"]
    plotdir = os.path.join("runs","plots", base); os.makedirs(plotdir, exist_ok=True)
    savedir = os.path.join("runs","models", base); os.makedirs(savedir, exist_ok=True)

    # geometry
    geom = make_wavegeom(cfg)
    nx, ny = cfg["grid"]["nx"], cfg["grid"]["ny"]
    dx, dy = cfg["grid"]["dx_m"], cfg["grid"]["dy_m"]

# sources & probes
src = build_sources(cfg, nx, ny, dx, dy)
probes = build_disk_probes(cfg, nx, ny)

# solver
dt = cfg["time"]["dt_s"]; timesteps = cfg["time"]["timesteps"]
model = spintorch.MMSolver(geom, dt, src, probes).to(dev)
model.retain_history = bool(cfg.get("save_history", True))

# temporal drive
X, t = temporal_envelope(cfg, dt, timesteps, dev)
INPUTS = X.repeat(1, 1, len(src))   # [1, T, Nsrc]

# Optional: baseline assertion there are no extra fields
try:
    from src.analysis_helpers import assert_baseline_no_weights
    assert_baseline_no_weights(model)
except Exception:
    pass

# Plot temporal pulse (for the run log)
t_ns = t.squeeze().cpu().numpy()*1e9
env = X.squeeze().detach().cpu().numpy()
plt.figure(); plt.plot(t_ns, env); plt.xlabel("Time (ns)"); plt.ylabel("B_exc (T)")
plt.title("Temporal excitation (solver-only)"); plt.grid(True)
plt.savefig(os.path.join(plotdir, "temporal_envelope_solver_only.png"), dpi=300)

# Run once with the pulse
_ = solve_once(model, INPUTS, savedir, plotdir, cfg, tag="pulse")

# (Optional) Run once with zero drive for a dark reference
X0 = torch.zeros_like(X)
INPUTS0 = X0.repeat(1, 1, len(src))
_ = solve_once(model, INPUTS0, savedir, plotdir, cfg, tag="dark")

# mz_full = torch.stack(model.m_history, 1)[0, :, 2, ] - model.m0[0, 2, ].unsqueeze(0).cpu()
print(np.shape(model.m_history))

In [3]:
import torch, matplotlib.pyplot as plt
import os
import numpy as np
import os, torch
import matplotlib.pyplot as plt
import spintorch
from src.cfg import load_configs
from src.geometry import make_wavegeom
from src.sources import build_sources, temporal_envelope
from src.readouts_legacy import build_disk_probes
from src.solver_only import solve_once
from src.readout import ROIIntegrator

Ms_scales = np.linspace(1, 0.9, 20)
f0 = 5e9  # Hz

# def main():
cfg = load_configs(".") 
assert cfg.get("run_mode","solver_only") == "solver_only", \
    "Set run_mode to 'solver_only' in configs/sim_defaults.json"

dev = torch.device(cfg.get("device","cuda"))
print("Running solver-only on", dev)

# I/O
base = cfg["io"]["basedir"]
plotdir = os.path.join("runs","plots", base); os.makedirs(plotdir, exist_ok=True)
savedir = os.path.join("runs","models", base); os.makedirs(savedir, exist_ok=True)

nx, ny = cfg["grid"]["nx"], cfg["grid"]["ny"]
dx, dy = cfg["grid"]["dx_m"], cfg["grid"]["dy_m"]
dt, timesteps = cfg["time"]["dt_s"], cfg["time"]["timesteps"]

single_source = {
    "Bt_T": 1.0e-3,
    "ports_fractional_y": [0.5],
    "x0_m": 5.0e-7,
    "sigma_x_m": 1.0e-7,
    "sigma_y_m": 1.0e-7,
    "temporal": {
      "type": "gaussian",
      "center_frac": 0.3,
      "sigma_frac": 0.2
    }
}

cfg["source"] = single_source

# Source: Gaussian at left edge
src = build_sources(cfg, nx, ny, dx, dy)
probes = build_disk_probes(cfg, nx, ny)

# --- define ROI ---
# Example: a small rectangle near the right edge
roi = ROIIntegrator(
    nx, ny, dx, dy,
    t_start_idx=0, t_end_idx=timesteps, dt_s = dt,
    box_xy=[nx*dx*0.8, nx*dx*0.9, ny*dy*0.45, ny*dy*0.55],  # meters
    device=dev
)

results = {}
for scale in Ms_scales:
    # --- override scatterer config ---
    cfg_scatt = {
        "enabled": True,
        "mode": "single",
        "Ms_scale": scale,
        "diameter_m": 200e-9,
        "single_center_m": [0.3*nx*dx, ny*dy/2]
    }
    cfg["scatterers"] = cfg_scatt

    # Geometry
    geom = make_wavegeom(cfg)
    model = spintorch.MMSolver(geom, dt, src, probes).to(dev)
    model.retain_history = True

    # Temporal drive
    X, t = temporal_envelope(cfg, dt, timesteps, dev)
    INPUTS = X.repeat(1, 1, len(src))

    with torch.no_grad():
        u = model(INPUTS).sum(dim=1)

    # --- ROI readout ---
    out_val = roi(torch.stack(model.m_history, 1).to(dev))

    # FFT of ROI signal
    # sig = u[0].cpu().numpy()  # pick probe at transmission side
    sig = out_val.detach().cpu().numpy() # ROI
    freqs = np.fft.rfftfreq(len(sig), d=dt)
    SPEC = np.fft.rfft(sig - sig.mean())
    idx = np.argmin(np.abs(freqs - f0))
    amp, phase = np.abs(SPEC[idx]), np.angle(SPEC[idx])
    results[scale] = (amp, phase)

    # --- Snapshots of mz with ROI overlay ---
    snapshot_times = np.linspace(0, timesteps-1, 5, dtype=int)
    for j, t_idx in enumerate(snapshot_times):
        mz2d = model.m_history[t_idx][0,2,:,:].cpu().numpy() # 
        plt.figure(figsize=(5,4))
        plt.imshow(mz2d.T, origin="lower", cmap="RdBu",
                    extent=[0, nx*dx*1e6, 0, ny*dy*1e6])
        plt.colorbar(label=r"$m_z$")
        plt.title(f"Ms_scale={scale:.2f}, t={t_idx*dt*1e12:.1f} ps")

        # highlight ROI
        x0, x1, y0, y1 = [nx*dx*0.8, nx*dx*0.9, ny*dy*0.45, ny*dy*0.55]
        rect = plt.Rectangle((x0*1e6, y0*1e6),
                                (x1-x0)*1e6, (y1-y0)*1e6,
                                linewidth=2, edgecolor='yellow', facecolor='none')
        plt.gca().add_patch(rect)

        # highlight the scatterer itself
        circ = plt.Circle((cfg_scatt["single_center_m"][0]*1e6, cfg_scatt["single_center_m"][1]*1e6), radius = cfg_scatt["diameter_m"]/2*1e6, 
                                linewidth=2, edgecolor='red', facecolor='none')
        plt.gca().add_patch(circ)

        plt.xlabel("x (µm)"); plt.ylabel("y (µm)")
        plt.tight_layout()
        plt.savefig(os.path.join(plotdir, f"mz_snapshot_scale{scale:.2f}_t{j}.png"), dpi=200)
        plt.show()

Running solver-only on cuda


c:\Users\Tojo\spintorch\source.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('x', torch.tensor(x, dtype=torch.int64))
c:\Users\Tojo\spintorch\source.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('y', torch.tensor(y, dtype=torch.int64))
c:\Users\Tojo\lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differe

we need more theory than just "one is linear, the other is quadratic, trust me bro

also, dodn't forget to fix ROI!

In [4]:
amps, phases = zip(*[results[s] for s in Ms_scales])

koeff_fitted_amp = np.polyfit(Ms_scales, amps, 2)
fitted_amp = koeff_fitted_amp[2] + koeff_fitted_amp[1]*Ms_scales + koeff_fitted_amp[0]*Ms_scales*Ms_scales
koeff_fitted_phases = np.polyfit(Ms_scales, np.unwrap(phases), 1)
fitted_phases = koeff_fitted_phases[1] + koeff_fitted_phases[0]*Ms_scales

_diam = str(cfg["scatterers"]["diameter_m"])
_pos = str([i*1e-7 for i in np.round([ii*1e6 for ii in cfg["scatterers"]["single_center_m"]], 2)])

plt.figure()
plt.plot(Ms_scales, amps, 'o-')
plt.plot(Ms_scales, fitted_amp, '--')
plt.xlabel("Ms scale"); plt.ylabel("Transmission amplitude")
plt.title(_diam + " " + _pos + " 09-1 ROI (single source) scatterer attenuation vs Ms scale")
plt.savefig(os.path.join(plotdir, _diam + _pos + " 09-1 ROI (single source) scatterer attenuation vs Ms scale.png"), dpi=200)
    
plt.figure()
plt.plot(Ms_scales, np.unwrap(phases), 'o-')
plt.plot(Ms_scales, fitted_phases, '--')
plt.xlabel("Ms scale"); plt.ylabel("Phase (rad)")
plt.title(_diam + " " + _pos + " 09-1 ROI (single source) scatterer phase shift vs Ms scale")
plt.savefig(os.path.join(plotdir, _diam + " " + _pos + " 09-1 ROI (single source) scatterer phase shift vs Ms scale.png"), dpi=200)
plt.show()

# # --- ridge vs theory error ---
# power_per_k = PSD_x.sum(axis=0)
# valid = power_per_k >= np.percentile(power_per_k, 75)
# peak_idx = PSD_x[:, valid].argmax(axis=0)
# f_peak = freq_GHz_x[peak_idx]
# k_sel = kx[valid]

# f_sel_th = omega_BV(k_sel, H0, Ms, gamma, tfilm)/(2*np.pi)/1e9

# rel_err = np.abs(f_peak - f_sel_th)/np.maximum(f_sel_th, 1e-12)
# print(f"mean rel. error = {100*rel_err.mean():.2f}%   (pass if < 10%)")
# print(f"max  rel. error = {100*rel_err.max():.2f}%   on {valid.sum()} k-columns")

C:\Users\Tojo\AppData\Local\Temp\ipykernel_9940\2059109259.py:24: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [22]:
import torch, matplotlib.pyplot as plt
import numpy as np
import os, torch
import matplotlib.pyplot as plt
import spintorch
from src.cfg import load_configs
from src.geometry import make_wavegeom
from src.sources import build_sources, temporal_envelope
from src.readouts_legacy import build_disk_probes
from src.solver_only import solve_once
from src.readout import ROIIntegrator

f0 = 5e9  # Hz

# def main():
cfg = load_configs(".") 
assert cfg.get("run_mode","solver_only") == "solver_only", \
    "Set run_mode to 'solver_only' in configs/sim_defaults.json"

dev = torch.device(cfg.get("device","cuda"))
print("Running solver-only on", dev)

# I/O
base = cfg["io"]["basedir"]
plotdir = os.path.join("runs","plots", base); os.makedirs(plotdir, exist_ok=True)
savedir = os.path.join("runs","models", base); os.makedirs(savedir, exist_ok=True)

nx, ny = cfg["grid"]["nx"], cfg["grid"]["ny"]
dx, dy = cfg["grid"]["dx_m"], cfg["grid"]["dy_m"]
dt, timesteps = cfg["time"]["dt_s"], cfg["time"]["timesteps"]

double_source = {
    "Bt_T": 1.0e-3,
    "ports_fractional_y": [0.4, 0.6],
    "x0_m": 5.0e-7,
    "sigma_x_m": 1.0e-7,
    "sigma_y_m": 1.0e-7,
    "temporal": {
      "type": "gaussian",
      "center_frac": 0.3,
      "sigma_frac": 0.2
    }
}

cfg["source"] = double_source

two_probes =  { "Ndisk": 2, "x_index": -15, "radius_cells": 2 }

cfg["probes"] = two_probes

# Source: Gaussian at left edge
src = build_sources(cfg, nx, ny, dx, dy)
probes = build_disk_probes(cfg, nx, ny)

# --- define ROI ---
# Example: a small rectangle near the right edge
roi = ROIIntegrator(
    nx, ny, dx, dy,
    t_start_idx=0, t_end_idx=timesteps, dt_s = dt,
    box_xy=[nx*dx*0.8, nx*dx*0.9, ny*dy*0.45, ny*dy*0.55],  # meters
    device=dev
)

results = {}

scale = 0.9

# --- override scatterer config ---
cfg_scatt = {
    "enabled": True,
    "mode": "grid",
    "Ms_scale": scale,
    "diameter_m": 200e-9,
    "scattering_mask_shift": [200, 0],
    "pitch_m": 5e-7,
    "nx_span_m": 5e-6,
    "ny_span_m": 5e-6
}
cfg["scatterers"] = cfg_scatt

# Geometry
geom = make_wavegeom(cfg)
model = spintorch.MMSolver(geom, dt, src, probes).to(dev)
model.retain_history = True

# Temporal drive
X, t = temporal_envelope(cfg, dt, timesteps, dev)
INPUTS = X.repeat(1, 1, len(src))

with torch.no_grad():
    u = model(INPUTS).sum(dim=1)

# --- ROI readout ---
out_val = roi(torch.stack(model.m_history, 1).to(dev))

# FFT of ROI signal
sig = u[0].cpu().numpy()  # pick probe at transmission side
# sig = out_val.detach().cpu().numpy() # ROI
freqs = np.fft.rfftfreq(len(sig), d=dt)
SPEC = np.fft.rfft(sig - sig.mean())
idx = np.argmin(np.abs(freqs - f0))
amp, phase = np.abs(SPEC[idx]), np.angle(SPEC[idx])
results[scale] = (amp, phase)

# --- Snapshots of mz with ROI overlay ---
snapshot_times = np.linspace(0, timesteps-1, 5, dtype=int)
for j, t_idx in enumerate(snapshot_times):
    print("timestep " + str(j))
    mz2d = model.m_history[t_idx][0,2,:,:].cpu().numpy() # 
    plt.figure(figsize=(5,4))
    plt.imshow(mz2d.T, origin="lower", cmap="RdBu",
                extent=[0, nx*dx*1e6, 0, ny*dy*1e6])
    plt.colorbar(label=r"$m_z$")
    plt.title(f"Ms_scale={scale:.2f}, t={t_idx*dt*1e12:.1f} ps")

    # highlight ROI
    x0, x1, y0, y1 = [nx*dx*0.8, nx*dx*0.9, ny*dy*0.45, ny*dy*0.55]
    rect = plt.Rectangle((x0*1e6, y0*1e6),
                            (x1-x0)*1e6, (y1-y0)*1e6,
                            linewidth=2, edgecolor='yellow', facecolor='none')
    plt.gca().add_patch(rect)

    if cfg_scatt["enabled"] and cfg_scatt["mode"] != "single":
        diam = cfg_scatt["diameter_m"] * 1e6    # µm
        pitch = cfg_scatt["pitch_m"] * 1e6      # µm
        nx_span = cfg_scatt["nx_span_m"] * 1e6  # µm
        ny_span = cfg_scatt["ny_span_m"] * 1e6  # µm

        # array center (in µm) – place in the middle of the spans
        x0 = nx_span/2
        y0 = ny_span/2

        # how many scatterers along each axis
        ncols = int(nx_span // pitch)
        nrows = int(ny_span // pitch)

        ax = plt.gca()
        for i in range(ncols):
            for k in range(nrows):
                cx = x0 - (nx_span/2) + i * pitch + pitch/2
                cy = y0 - (ny_span/2) + k * pitch + pitch/2
                circ = plt.Circle((cx, cy),
                                radius=diam/2,
                                linewidth=1.2,
                                edgecolor='red',
                                facecolor='none')
                ax.add_patch(circ)

    plt.xlabel("x (µm)"); plt.ylabel("y (µm)")
    plt.tight_layout()
    plt.savefig(os.path.join(plotdir, f"mz_snapshot_scale{scale:.2f}_t{j}.png"), dpi=200)
    plt.show()



Running solver-only on cuda
timestep 0


C:\Users\Tojo\AppData\Local\Temp\ipykernel_9940\656218429.py:152: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


timestep 1
timestep 2
timestep 3
timestep 4


In [19]:
cfg["scatterers"].get("scattering_mask_shift")[0]

20